# Model v1 (simplified) - CatBoost baseline for VIEWS forecasting


## 0. Setup

**What we do:** import libraries and define paths.  
**Outcome:** stable execution across machines (paths can be overridden via env vars).

In [118]:
import os
import json
from datetime import timedelta

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
import os
import sys

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
sys.path.append('..')
base_dir = os.path.abspath(os.path.join("..", ".."))
print(base_dir)


# Paths (override if needed)
ALLDATA_PATH = os.path.join(base_dir, "data", "AllData.csv")
TESTDATA_PATH = os.path.join(base_dir, "data", "TestDataset.csv")  # optional
ARTIFACTS_DIR = os.getenv("ARTIFACTS_DIR", "artifacts")
OUTPUTS_DIR = os.getenv("OUTPUTS_DIR", "outputs")
HOLDOUT_DAYS = int(os.getenv("HOLDOUT_DAYS", "30"))
MODEL_TAG = os.getenv("MODEL_TAG", "v1_simplified")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("ALLDATA_PATH:", ALLDATA_PATH)
print("TESTDATA_PATH:", TESTDATA_PATH)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("HOLDOUT_DAYS:", HOLDOUT_DAYS)
print("MODEL_TAG:", MODEL_TAG)

/Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster
ALLDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/AllData.csv
TESTDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/TestDataset.csv
ARTIFACTS_DIR: artifacts
OUTPUTS_DIR: outputs
HOLDOUT_DAYS: 30
MODEL_TAG: v1_simplified


In [119]:
test_df = pd.read_csv(TESTDATA_PATH)
test_df.head()

,CPM,CHANNEL_NAME,DATE,VIEWS
0,25.0,topcareerschool,2023-03-11,NaN
1,10.0,topcareerschool,2023-03-11,NaN
2,13.0,hrsecrets_life,2023-03-11,NaN
3,7.0,hrsecrets_life,2023-03-11,NaN
4,7.0,Dirclub,2023-03-12,NaN


## 1. Load data and diagnostics

**What we do:** read `AllData.csv`, normalize column names, parse `DATE`, check data quality.  
**Outcome:** a clean `df` with diagnostics to identify potential issues.

In [120]:
df = pd.read_csv(ALLDATA_PATH)

# Dataset quirk: sometimes columns have leading/trailing spaces
df.columns = df.columns.str.strip()

print(f"Initial shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Parse DATE
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

required = {"CPM", "CHANNEL_NAME", "DATE", "VIEWS"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

# Data quality checks
print("\n=== Data Quality Checks ===")
print(f"Total rows: {len(df):,}")
print(f"Missing DATE: {df['DATE'].isna().sum()}")
print(f"Missing VIEWS: {df['VIEWS'].isna().sum()}")
print(f"Missing CPM: {df['CPM'].isna().sum()}")
print(f"Missing CHANNEL_NAME: {df['CHANNEL_NAME'].isna().sum()}")

# Remove rows with missing critical data
initial_len = len(df)
df = df.dropna(subset=["DATE", "VIEWS", "CPM", "CHANNEL_NAME"]).copy()
print(f"Rows after cleaning: {len(df):,} (removed {initial_len - len(df)})")

# VIEWS distribution diagnostics
print("\n=== VIEWS Distribution ===")
print(df["VIEWS"].describe())
print(f"VIEWS == 0: {(df['VIEWS'] == 0).sum()} ({(df['VIEWS'] == 0).mean()*100:.2f}%)")
print(f"VIEWS < 0: {(df['VIEWS'] < 0).sum()}")
print(f"VIEWS > 100000: {(df['VIEWS'] > 100000).sum()} ({(df['VIEWS'] > 100000).mean()*100:.2f}%)")

# CPM distribution diagnostics
print("\n=== CPM Distribution ===")
print(df["CPM"].describe())
print(f"CPM <= 0: {(df['CPM'] <= 0).sum()}")
print(f"CPM > 100: {(df['CPM'] > 100).sum()} ({(df['CPM'] > 100).mean()*100:.2f}%)")

# Date range
print("\n=== Date Range ===")
print(f"Min date: {df['DATE'].min()}")
print(f"Max date: {df['DATE'].max()}")
print(f"Days span: {(df['DATE'].max() - df['DATE'].min()).days}")
print(f"Unique dates: {df['DATE'].nunique()}")

# Channel statistics
print("\n=== Channel Statistics ===")
print(f"Unique channels: {df['CHANNEL_NAME'].nunique()}")
print(f"Channels with < 10 samples: {(df['CHANNEL_NAME'].value_counts() < 10).sum()}")
print(f"Top 10 channels by frequency:")
print(df['CHANNEL_NAME'].value_counts().head(10))

# Check for potential issues
print("\n=== Potential Issues ===")
if (df['VIEWS'] == 0).mean() > 0.5:
    print("⚠️  WARNING: More than 50% of VIEWS are zero - model may struggle")
if (df['VIEWS'] < 0).any():
    print("⚠️  WARNING: Negative VIEWS found - will clip to 0")
if (df['CPM'] <= 0).any():
    print("⚠️  WARNING: Non-positive CPM found")
if df['CHANNEL_NAME'].isna().any():
    print("⚠️  WARNING: Missing channel names found")

df.head()

Initial shape: (142609, 7)
Columns: ['AD_ID', 'CPM', 'VIEWS', 'CLICKS', 'ACTIONS', 'CHANNEL_NAME', 'DATE']

=== Data Quality Checks ===
Total rows: 142,609
Missing DATE: 0
Missing VIEWS: 0
Missing CPM: 0
Missing CHANNEL_NAME: 0
Rows after cleaning: 142,609 (removed 0)

=== VIEWS Distribution ===
count    1.426090e+05
mean     9.998400e+02
std      7.260922e+03
min      0.000000e+00
25%      6.500000e+01
50%      2.550000e+02
75%      6.620000e+02
max      1.136470e+06
Name: VIEWS, dtype: float64
VIEWS == 0: 42 (0.03%)
VIEWS < 0: 0
VIEWS > 100000: 73 (0.05%)

=== CPM Distribution ===
count    142609.000000
mean          8.994971
std          29.404112
min           1.000000
25%           2.000000
50%           3.530000
75%           8.300000
max         999.910000
Name: CPM, dtype: float64
CPM <= 0: 0
CPM > 100: 743 (0.52%)

=== Date Range ===
Min date: 2024-10-02 00:00:00
Max date: 2025-12-17 00:00:00
Days span: 441
Unique dates: 430

=== Channel Statistics ===
Unique channels: 35957
C

,AD_ID,CPM,VIEWS,CLICKS,ACTIONS,CHANNEL_NAME,DATE
0,3652,1.2,52,0,0,tanya_in_france,2024-10-02
1,3653,1.2,54,2,0,relocator_cc,2024-10-02
2,3654,1.2,15,1,0,teleportazia,2024-10-02
3,3655,1.5,85,2,1,spetsialist_visa_support,2024-10-02
4,3656,19.4,688,21,3,pitkvch_news,2024-10-02


## 2. Time-based split (holdout last N days)

**What we do:** split by time for honest evaluation.  
**Why:** random split leaks time patterns.  
**Outcome:** `train_df` and `valid_df` for local scoring.

In [121]:
def split_last_days(data: pd.DataFrame, holdout_days: int = 30):
    unique_dates = np.sort(data["DATE"].unique())
    if len(unique_dates) <= holdout_days:
        raise ValueError(
            f"Not enough unique dates ({len(unique_dates)}) for holdout_days={holdout_days}"
        )
    cutoff = unique_dates[-holdout_days]
    train = data.loc[data["DATE"] < cutoff].copy()
    valid = data.loc[data["DATE"] >= cutoff].copy()
    return train, valid, cutoff, unique_dates[-1]

train_df, valid_df, cutoff, dmax = split_last_days(df, holdout_days=HOLDOUT_DAYS)

print("Train:", train_df.shape, "Valid:", valid_df.shape)
print("Cutoff:", cutoff, "→", dmax)


Train: (131923, 7) Valid: (10686, 7)
Cutoff: 2025-11-18T00:00:00.000000000 → 2025-12-17T00:00:00.000000000


## 3. Feature engineering (minimal)

We build a compact feature set focused on stable signal and interpretability.

**CPM features:**
- `log_cpm` - log1p(CPM) to handle heavy tails
- `cpm_to_ch_median` - CPM relative to the channel median

**Channel features:**
- `CHANNEL_NAME` (categorical)
- `ch_med_smooth` - shrinked median of log1p(VIEWS) per channel
- `ch_count_log` - log1p(number of rows per channel)

**Date features (yearless seasonality):**
- `dow`, `is_weekend`
- `month`
- `doy_sin`, `doy_cos`


In [122]:
# Use yearless seasonality features (safe for 2023 dates in test)

def add_date_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    d = out["DATE"]

    out["dow"] = d.dt.dayofweek.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)

    out["month"] = d.dt.month.astype(int)
    out["dayofyear"] = d.dt.dayofyear.astype(int)

    # Cyclic encoding for day-of-year
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)

    return out


def fit_preprocess_params(train: pd.DataFrame) -> dict:
    # Reserved for future use (kept for consistency)
    return {}


def apply_basic_preprocess(data: pd.DataFrame, params=None) -> pd.DataFrame:
    out = data.copy()

    # CPM features
    out["cpm"] = out["CPM"].astype(float)
    out["log_cpm"] = np.log1p(out["cpm"].clip(lower=0))

    # DATE features
    out = add_date_features(out)

    return out


def fit_channel_stats(train: pd.DataFrame, alpha: float = 10.0):
    # Robust channel stats on train only (log scale), with shrinkage.
    y = np.log1p(train["VIEWS"].clip(lower=0))
    global_med = float(np.median(y))

    stats = (
        train.assign(y=y)
             .groupby("CHANNEL_NAME")["y"]
             .agg(ch_count="size", ch_med="median")
             .reset_index()
    )

    w = stats["ch_count"] / (stats["ch_count"] + alpha)
    stats["ch_med_smooth"] = w * stats["ch_med"] + (1 - w) * global_med

    return stats[["CHANNEL_NAME", "ch_count", "ch_med_smooth"]], global_med


def apply_channel_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_med: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_count"] = out["ch_count"].fillna(0).astype(int)
    out["ch_count_log"] = np.log1p(out["ch_count"])
    out["ch_med_smooth"] = out["ch_med_smooth"].fillna(global_med).astype(float)
    return out


def fit_channel_cpm_stats(train: pd.DataFrame):
    stats = (
        train.groupby("CHANNEL_NAME")["CPM"]
             .median()
             .reset_index(name="ch_cpm_median")
    )
    global_median = float(train["CPM"].median())
    return stats, global_median


def apply_channel_cpm_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_median: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_cpm_median"] = out["ch_cpm_median"].fillna(global_median)
    denom = out["ch_cpm_median"].replace(0, np.nan)
    out["cpm_to_ch_median"] = (out["cpm"] / denom).fillna(1.0)
    return out


## 4. Build train/valid matrices

**What we do:** fit preprocessing on train only, build features, create CatBoost Pools.  
**Outcome:** `train_pool`, `valid_pool` and feature specification.

In [123]:
# Fit preprocess on train only (no leakage)
preprocess = fit_preprocess_params(train_df)

# Basic features
train_fe = apply_basic_preprocess(train_df, preprocess)
valid_fe = apply_basic_preprocess(valid_df, preprocess)

# Channel stats (train only)
ch_stats, global_med_log = fit_channel_stats(train_df, alpha=10.0)
train_fe = apply_channel_stats(train_fe, ch_stats, global_med_log)
valid_fe = apply_channel_stats(valid_fe, ch_stats, global_med_log)

# Channel CPM stats (train only)
ch_cpm_stats, global_cpm_median = fit_channel_cpm_stats(train_df)
train_fe = apply_channel_cpm_stats(train_fe, ch_cpm_stats, global_cpm_median)
valid_fe = apply_channel_cpm_stats(valid_fe, ch_cpm_stats, global_cpm_median)

# Feature spec (minimal and stable)
feature_cols = [
    "log_cpm",
    "cpm_to_ch_median",
    "dow", "is_weekend", "month", "doy_sin", "doy_cos",
    "ch_count_log", "ch_med_smooth",
    "CHANNEL_NAME",  # categorical
]
cat_features = ["CHANNEL_NAME"]

X_train = train_fe[feature_cols]
X_valid = valid_fe[feature_cols]

y_train_raw = train_df["VIEWS"].astype(float).values
y_valid_raw = valid_df["VIEWS"].astype(float).values

y_train = np.log1p(np.clip(y_train_raw, 0, None))
y_valid = np.log1p(np.clip(y_valid_raw, 0, None))

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

X_train.head()


,log_cpm,cpm_to_ch_median,dow,is_weekend,month,doy_sin,doy_cos,ch_count_log,ch_med_smooth,CHANNEL_NAME
0,0.788457,0.666667,2,0,10,-0.999668,0.025748,2.639057,5.163988,tanya_in_france
1,0.788457,0.217195,2,0,10,-0.999668,0.025748,2.944439,5.249288,relocator_cc
2,0.788457,0.150000,2,0,10,-0.999668,0.025748,3.401197,4.984407,teleportazia
3,0.916291,1.000000,2,0,10,-0.999668,0.025748,2.302585,5.046654,spetsialist_visa_support
4,3.015535,4.850000,2,0,10,-0.999668,0.025748,2.302585,5.881408,pitkvch_news


## 5. Train CatBoost (v1 simplified)


In [124]:
# Helper function for metrics calculation
def calculate_metrics(y_true, y_pred, name=""):
    # Calculate comprehensive metrics.
    y_true = np.asarray(y_true, dtype=float).clip(min=0)
    y_pred = np.asarray(y_pred, dtype=float).clip(min=0)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    # RMSLE
    rmsle = float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

    # SMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1, denom)
    smape = float(np.mean(np.abs(y_pred - y_true) / denom)) * 100

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "RMSLE": rmsle,
        "SMAPE": smape
    }

    if name:
        print(f"
{name} Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

    return metrics

print("=" * 70)
print("Training CatBoost on log1p(VIEWS)")
print("=" * 70)

model = CatBoostRegressor(
    loss_function="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=4000,
    l2_leaf_reg=5,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.8,
    random_seed=RANDOM_SEED,
    eval_metric="RMSE",
    verbose=500,
    od_type="Iter",
    od_wait=200,
    task_type="CPU",
    devices="0",
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=500)

# Predictions on validation (convert back to raw scale)
pred_valid_log = model.predict(valid_pool)
pred_valid = np.clip(np.expm1(pred_valid_log), 0, None)

metrics_valid = calculate_metrics(y_valid_raw, pred_valid, "CatBoost (log1p target)")


SyntaxError: EOL while scanning string literal (1721340848.py, line 26)

## 6. Evaluate (raw scale)

We predict in log space and invert back with `expm1`.  
Outcome: local metrics you can compare to baselines and track between versions.

In [ ]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true, dtype=float), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom))

metrics_local = {
    "MAE": mae(y_valid_raw, pred_valid),
    "RMSE": rmse(y_valid_raw, pred_valid),
    "RMSLE": rmsle(y_valid_raw, pred_valid),
    "SMAPE": smape(y_valid_raw, pred_valid),
}
metrics_local


## 7. Baseline comparison (same split)

Outcome: sanity check. The simplified model should beat these baselines.


In [ ]:
global_median = float(train_df["VIEWS"].median())

# Baseline 1: global median
b1 = np.full_like(y_valid_raw, fill_value=global_median, dtype=float)

# Baseline 2: DOW median
train_dow = train_df["DATE"].dt.dayofweek
valid_dow = valid_df["DATE"].dt.dayofweek
dow_median = train_df.groupby(train_dow)["VIEWS"].median()
b2 = valid_dow.map(dow_median).fillna(global_median).to_numpy(dtype=float)

# Baseline 3: channel shrink baseline (raw scale)
alpha = 5
ch_raw = (train_df.groupby("CHANNEL_NAME")["VIEWS"].agg(ch_median="median", ch_count="size").reset_index())
valid_ch = valid_df[["CHANNEL_NAME"]].merge(ch_raw, on="CHANNEL_NAME", how="left")
w = (valid_ch["ch_count"] / (valid_ch["ch_count"] + alpha)).fillna(0.0)
b3 = (w * valid_ch["ch_median"].fillna(global_median) + (1 - w) * global_median).to_numpy(dtype=float)

def score_row(name, pred):
    return {
        "model": name,
        "MAE": mae(y_valid_raw, pred),
        "RMSE": rmse(y_valid_raw, pred),
        "RMSLE": rmsle(y_valid_raw, pred),
        "SMAPE": smape(y_valid_raw, pred),
    }

rows = [
    score_row("baseline_global_median", b1),
    score_row("baseline_dow_median", b2),
    score_row("baseline_channel_shrink", b3),
    score_row("catboost_v1_simplified", pred_valid),
]
pd.DataFrame(rows).sort_values("MAE")


## 8. Final training on FULL data (for submission)

**Important:** once you selected a configuration, train on **all** rows to avoid losing the last N days.  
Outcome: `final_model` + full artifacts for submission and deployment.

In [ ]:
full_df = df.copy()

# Fit preprocessing on FULL data
preprocess_full = fit_preprocess_params(full_df)
full_fe = apply_basic_preprocess(full_df, preprocess_full)

# Fit channel stats on FULL data (still offline, no leakage for submission)
ch_stats_full, global_med_log_full = fit_channel_stats(full_df, alpha=10.0)
full_fe = apply_channel_stats(full_fe, ch_stats_full, global_med_log_full)

# Fit channel CPM stats on FULL data
ch_cpm_stats_full, global_cpm_median_full = fit_channel_cpm_stats(full_df)
full_fe = apply_channel_cpm_stats(full_fe, ch_cpm_stats_full, global_cpm_median_full)

X_full = full_fe[feature_cols]
y_full_raw = full_df["VIEWS"].astype(float).values
y_full_log = np.log1p(np.clip(y_full_raw, 0, None))

full_pool = Pool(X_full, y_full_log, cat_features=cat_features)

final_model = CatBoostRegressor(**model.get_params())
final_model.fit(full_pool, verbose=200)


## 9. Save artifacts

Outcome: you can reproduce predictions in batch scripts and the API.

In [ ]:
# Save model
model_path = os.path.join(ARTIFACTS_DIR, f"model_{MODEL_TAG}.cbm")
final_model.save_model(model_path)

# Save preprocessing + channel stats references
with open(os.path.join(ARTIFACTS_DIR, f"preprocess_{MODEL_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump(preprocess_full, f, ensure_ascii=False, indent=2)

# Save channel stats table (CSV is easy to inspect)
ch_stats_path = os.path.join(ARTIFACTS_DIR, f"channel_stats_{MODEL_TAG}.csv")
ch_stats_full.to_csv(ch_stats_path, index=False)

# Save channel CPM stats
ch_cpm_stats_path = os.path.join(ARTIFACTS_DIR, f"channel_cpm_stats_{MODEL_TAG}.csv")
ch_cpm_stats_full.to_csv(ch_cpm_stats_path, index=False)

with open(os.path.join(ARTIFACTS_DIR, f"meta_{MODEL_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump({
        "created_at": "2026-01-10",
        "model": "CatBoostRegressor",
        "target": "log1p(VIEWS)",
        "feature_cols": feature_cols,
        "cat_features": cat_features,
        "holdout_days_for_eval": HOLDOUT_DAYS,
        "notes": "Simplified v1: yearless seasonality + log_cpm + channel stats + relative CPM",
    }, f, ensure_ascii=False, indent=2)

with open(os.path.join(ARTIFACTS_DIR, f"metrics_{MODEL_TAG}_local.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_local, f, ensure_ascii=False, indent=2)

print("Saved:", model_path)
print("Saved:", ch_stats_path)
print("Saved:", ch_cpm_stats_path)


## 10. Optional: fill `TestDataset.csv` and export a submission file

Outcome: `outputs/TestDataset_filled_model_<MODEL_TAG>.csv` ready for upload.


In [ ]:
if os.path.exists(TESTDATA_PATH):
    test_df = pd.read_csv(TESTDATA_PATH)
    test_df.columns = test_df.columns.str.strip()
    test_df["DATE"] = pd.to_datetime(test_df["DATE"], errors="coerce")

    test_fe = apply_basic_preprocess(test_df, preprocess_full)
    test_fe = apply_channel_stats(test_fe, ch_stats_full, global_med_log_full)
    test_fe = apply_channel_cpm_stats(test_fe, ch_cpm_stats_full, global_cpm_median_full)

    X_test = test_fe[feature_cols]
    test_pool = Pool(X_test, cat_features=cat_features)

    pred_test_raw = final_model.predict(test_pool)
    pred_test = np.clip(np.expm1(pred_test_raw), 0, None)

    out = test_df.copy()
    out["VIEWS"] = np.round(pred_test).astype(int)

    out_path = os.path.join(OUTPUTS_DIR, f"TestDataset_filled_model_{MODEL_TAG}.csv")
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
else:
    print(f"Test dataset not found at: {TESTDATA_PATH}")


## Recommended next step (Model v2)

If this simplified v1 still stalls:
- run a rolling backtest (3 windows)
- try MAE / Quantile objective
- optionally add offline TGStat/TGMaps channel features (subscribers, avg views, etc.) cached locally
